# ML Experiments — Testing Four Ideas Against the Baseline

Four independent additions to the existing EWMA + HRP + vol-target pipeline,
each evaluated **separately** against the same baseline so you can see which
(if any) are worth pursuing before touching `src/`:

1. **Vol forecasting** — ML model vs. EWMA for next-period volatility
2. **HRP bisection weighting** — learned top-level split ratio vs. the fixed inverse-variance formula
3. **Dynamic universe selection** — ML screen narrowing the universe per rebalance vs. the static 49-name list
4. **Hyperparameter search** — random search over existing `config.py` knobs vs. the hand-picked defaults

**Nothing in this notebook writes to `data/outputs/` or touches `src/`** — it's
read-only experimentation against the committed price cache.

## Leakage discipline

This repo's whole credibility rests on `tests/test_lookahead.py` proving zero
future-data leakage. Every experiment here follows the same rule, made explicit:
a single date, `decision_start_date`, marks the earliest point any real backtest
decision gets made. **All ML training data — features and labels alike — comes
strictly from before that date.** The backtests then run forward from that same
date, so a model never sees, even indirectly, the period it's being judged on.

This is a **first-pass exploratory** notebook: single train/holdout splits, not
walk-forward retraining, and short evaluation windows (a few years, not the
full ~11-year history) to keep iteration fast — the same window/speed tradeoff
already made in the dashboard's "Try Your Own" tab. Treat results here as
"is this worth building properly," not as production validation.


In [1]:
import sys, time, warnings
from pathlib import Path
from dataclasses import replace

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:.4f}".format

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "config.py").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import Config
from src.core.backtest import Strategy
from src.features import ewma_vol, ewmc_corr
from src.covariance import shrink_correlation, build_covariance, psd_repair
from src import hrp as hrp_module
from src.hrp import allocate
from src.benchmarks import benchmark_returns
from src.metrics import calculate_metrics, sharpe_ratio, max_drawdown, annualized_return, annualized_volatility, turnover

PRICES_PATH = PROJECT_ROOT / "data" / "processed" / "prices.parquet"
prices = pd.read_parquet(PRICES_PATH)
log_returns_full = np.log(prices / prices.shift(1)).dropna()
print(f"Universe: {prices.shape[1]} assets, {len(prices)} trading days, {prices.index.min().date()} to {prices.index.max().date()}")


Universe: 49 assets, 2889 trading days, 2015-01-01 to 2026-09-11


In [2]:
cfg = Config()

# --- the one leakage boundary every experiment below respects ---
EXPERIMENT_TEST_YEARS = 3            # how much recent history each experiment is judged on
TEST_ROWS = EXPERIMENT_TEST_YEARS * 252
WARMUP_ROWS = cfg.train_years * 252  # HRP's own rolling covariance warmup (matches config.py)

decision_start_idx = len(prices) - TEST_ROWS
decision_start_date = prices.index[decision_start_idx]

eval_slice = prices.iloc[decision_start_idx - WARMUP_ROWS:]           # warmup + test window every experiment backtests on
ml_train_prices = prices.iloc[:decision_start_idx]                     # everything strictly before the first real rebalance decision
ml_train_returns = np.log(ml_train_prices / ml_train_prices.shift(1)).dropna()

print(f"decision_start_date = {decision_start_date.date()}  (first date any model's OUTPUT gets used for a real decision)")
print(f"ML training data ends strictly before this date: {len(ml_train_returns)} rows available")
print(f"eval_slice: {len(eval_slice)} rows ({eval_slice.index.min().date()} to {eval_slice.index.max().date()}), "
      f"of which the last {TEST_ROWS} rows are where rebalances actually happen")


decision_start_date = 2023-08-22  (first date any model's OUTPUT gets used for a real decision)
ML training data ends strictly before this date: 2132 rows available
eval_slice: 1260 rows (2021-08-10 to 2026-09-11), of which the last 756 rows are where rebalances actually happen


## Shared walk-forward harness

A single reusable `walk_forward()` reimplements `Strategy.run()`'s loop
(smoothing, vol-target leverage, banded rebalancing, transaction costs —
byte-for-byte the same logic) but takes **pluggable hooks** for volatility
estimation, correlation estimation, allocation, and universe selection, each
defaulting to the production `src/` functions. That means:

- `walk_forward(eval_slice, cfg)` with no overrides should closely reproduce
  `Strategy(...).run(eval_slice)` — validated below as a sanity check before
  trusting anything built on top of it.
- Each experiment overrides exactly one hook, so any performance difference
  is attributable to that one change.


In [3]:
def walk_forward(prices_slice, config, vol_fn=None, corr_fn=None, allocate_fn=None, universe_fn=None, overlay_fn=None):
    """Reimplementation of Strategy.run() with pluggable estimation/allocation
    hooks, used only for notebook experimentation — src/core/backtest.py is
    untouched. overlay_fn(train_data, config) -> (vol_target, band), evaluated
    fresh each rebalance, lets vol_target/rebalance_band vary per-window
    instead of being fixed constants (used by Experiment 5)."""
    vol_fn = vol_fn or ewma_vol
    corr_fn = corr_fn or ewmc_corr
    allocate_fn = allocate_fn or allocate

    returns_df = np.log(prices_slice / prices_slice.shift(1)).dropna()
    all_cols = list(prices_slice.columns)

    train_window = config.train_years * 252
    test_window = config.test_months * 21
    n_steps = len(returns_df)
    start_idx = train_window

    rf_daily = config.risk_free_rate_annual / 252
    max_leverage = config.max_leverage
    weight_smoothing = config.weight_smoothing
    leverage_smoothing = config.leverage_smoothing
    transaction_cost = config.transaction_cost_bps / 10000

    strategy_returns, weight_history, cost_history = [], [], []
    hrp_weights = traded_weights = hrp_target = leverage_state = None

    for i in range(start_idx, n_steps, test_window):
        train_data_full = returns_df.iloc[i - train_window:i]
        test_end = min(i + test_window, n_steps)

        if overlay_fn:
            vol_target, band = overlay_fn(train_data_full, config)
        else:
            vol_target, band = config.vol_target_annual, config.rebalance_band

        universe = universe_fn(train_data_full, config, returns_df.index[i]) if universe_fn else all_cols
        train_data = train_data_full[universe]

        vol = vol_fn(train_data, span=config.ewma_span)
        corr = corr_fn(train_data, span=config.corr_span)
        corr = shrink_correlation(corr, config.corr_shrinkage)
        cov = build_covariance(vol, corr)
        cov = psd_repair(cov)

        candidate_dict = allocate_fn(cov, corr, config)
        candidate_hrp = pd.Series(candidate_dict, index=universe).reindex(all_cols).fillna(0.0)

        if hrp_target is None:
            hrp_target = candidate_hrp
        else:
            prev = hrp_target.reindex(all_cols).fillna(0.0)
            hrp_target = (1.0 - weight_smoothing) * prev + weight_smoothing * candidate_hrp
            total = hrp_target.sum()
            if total > 0:
                hrp_target = hrp_target / total

        leverage_raw = 1.0
        if vol_target and vol_target > 0:
            active = hrp_target.reindex(universe).fillna(0.0)
            port_vol = float(np.sqrt(active.values @ cov.values @ active.values)) if len(universe) else 0.0
            if port_vol > 0:
                leverage_raw = min(vol_target / port_vol, max_leverage)

        leverage_state = leverage_raw if leverage_state is None else (
            (1.0 - leverage_smoothing) * leverage_state + leverage_smoothing * leverage_raw)

        candidate_traded = hrp_target * leverage_state

        if traded_weights is None:
            adopt = True
        else:
            prev_traded = traded_weights.reindex(all_cols).fillna(0.0)
            adopt = float(np.abs(candidate_traded - prev_traded).sum()) > band

        cost_drag = 0.0
        if adopt:
            if traded_weights is not None:
                prev_traded = traded_weights.reindex(all_cols).fillna(0.0)
                cost_drag = float(np.abs(candidate_traded - prev_traded).sum()) * transaction_cost
            hrp_weights = hrp_target
            traded_weights = candidate_traded

        rebalance_date = returns_df.index[i]
        weight_history.append(pd.Series(hrp_weights, name=rebalance_date))

        active_traded = traded_weights.reindex(universe).fillna(0.0)
        cash_weight = 1.0 - float(traded_weights.sum())
        period_returns = returns_df.iloc[i:test_end][universe].dot(active_traded) + cash_weight * rf_daily
        if len(period_returns) > 0:
            period_returns = period_returns.copy()
            period_returns.iloc[0] -= cost_drag

        strategy_returns.append(period_returns)
        cost_history.append(cost_drag)

    strategy_series = pd.concat(strategy_returns)
    strategy_series.name = "Strategy"
    results = pd.DataFrame({"Strategy": strategy_series})
    weights_df = pd.DataFrame(weight_history).fillna(0.0)
    costs = pd.Series(cost_history, index=[w.name for w in weight_history])
    return results, weights_df, costs


def score(prices_slice, config, results, weights_df, costs, label, elapsed=None):
    lr = np.log(prices_slice / prices_slice.shift(1)).dropna()
    bench = benchmark_returns(lr, config)
    strat_returns = results["Strategy"].dropna()
    metrics = calculate_metrics(strat_returns, {k: v.dropna() for k, v in bench.items()}, weights_df, config, costs=costs)
    tag = f" ({elapsed:.1f}s)" if elapsed is not None else ""
    m = metrics["strategy"]
    print(f"[{label}]{tag} Sharpe {m['sharpe']:.3f} | MaxDD {m['max_drawdown']:.2%} | "
          f"AnnRet {m['annualized_return']:.2%} | AnnVol {m['annualized_volatility']:.2%} | Turnover {m['turnover']:.2%}")
    return metrics


def run_and_score(prices_slice, config, label, **hooks):
    t0 = time.time()
    results, weights_df, costs = walk_forward(prices_slice, config, **hooks)
    elapsed = time.time() - t0
    metrics = score(prices_slice, config, results, weights_df, costs, label, elapsed)
    return results, weights_df, metrics


In [4]:
# Sanity check: walk_forward() with all defaults should closely reproduce the real Strategy class
t0 = time.time()
real_strat = Strategy(train_window=cfg.train_years * 252, test_window=cfg.test_months * 21,
                       transaction_cost=cfg.transaction_cost_bps / 10000)
real_strat.config = cfg
real_results, real_weights, real_costs = real_strat.run(eval_slice)
real_metrics = score(eval_slice, cfg, real_results, real_weights, real_costs, "Strategy (production class)", time.time() - t0)

baseline_results, baseline_weights, baseline_metrics = run_and_score(eval_slice, cfg, "walk_forward() reimplementation, all defaults")

sharpe_diff = abs(real_metrics["strategy"]["sharpe"] - baseline_metrics["strategy"]["sharpe"])
assert sharpe_diff < 0.01, f"Reimplementation diverges from production Strategy by {sharpe_diff:.4f} Sharpe — investigate before trusting experiments below"
print(f"\nReimplementation matches production within {sharpe_diff:.4f} Sharpe — safe to build experiments on walk_forward().")


[Strategy (production class)] (14.4s) Sharpe 0.562 | MaxDD -11.56% | AnnRet 12.02% | AnnVol 10.72% | Turnover 0.29%


[walk_forward() reimplementation, all defaults] (14.4s) Sharpe 0.562 | MaxDD -11.56% | AnnRet 12.02% | AnnVol 10.72% | Turnover 0.29%

Reimplementation matches production within 0.0000 Sharpe — safe to build experiments on walk_forward().


## Baseline

The number every experiment below is measured against: current production
config (EWMA vol, fixed-formula HRP, static universe, hand-picked knobs) over
`eval_slice`'s test window.


In [5]:
BASELINE_SHARPE = baseline_metrics["strategy"]["sharpe"]
print(f"BASELINE Sharpe = {BASELINE_SHARPE:.3f}  <- every experiment below is compared to this")


BASELINE Sharpe = 0.562  <- every experiment below is compared to this


---
# Experiment 1: Vol Forecasting (ML vs. EWMA)

**Idea:** replace EWMA volatility with an ML model that predicts next-period
(21-day forward) realized volatility per asset, using only backward-looking
features. Vol is more forecastable than returns and this stays entirely
within the existing "risk-only" philosophy — HRP still never sees a return
forecast, just a (hopefully better) risk estimate.

**Training labels are built only from `ml_train_returns`** (strictly before
`decision_start_date`), then the fitted model is plugged into `walk_forward()`
as `vol_fn` and run on `eval_slice` — so every rebalance where the model's
prediction actually drives a decision falls after the data it was trained on.


In [6]:
FEATURE_WINDOWS = [5, 10, 20, 60]

def build_vol_features(returns_df):
    frames = []
    for w in FEATURE_WINDOWS:
        f = returns_df.rolling(w).std() * np.sqrt(252)
        frames.append(f.rename_axis(index="date", columns="asset").stack().rename(f"rvol_{w}"))
    frames.append(returns_df.abs().rename_axis(index="date", columns="asset").stack().rename("abs_ret_1"))
    frames.append((returns_df ** 2).rename_axis(index="date", columns="asset").stack().rename("sq_ret_1"))
    return frames

# forward-looking label: realized vol over the NEXT 21 trading days (what EWMA is implicitly trying to estimate)
forward_vol_label = (ml_train_returns.rolling(21).std().shift(-21) * np.sqrt(252)).rename_axis(index="date", columns="asset").stack().rename("label")
ewma_baseline_col = (ml_train_returns.ewm(span=cfg.ewma_span, adjust=False).std() * np.sqrt(252)).rename_axis(index="date", columns="asset").stack().rename("ewma_baseline")

vol_df = pd.concat(build_vol_features(ml_train_returns) + [ewma_baseline_col, forward_vol_label], axis=1).dropna().reset_index()
FEATURE_COLS = [c for c in vol_df.columns if c not in ("date", "asset", "label", "ewma_baseline")]

unique_dates = sorted(vol_df["date"].unique())
split_i = int(len(unique_dates) * 0.7)
train_dates, holdout_dates = set(unique_dates[:split_i]), set(unique_dates[split_i:])
train_rows = vol_df[vol_df["date"].isin(train_dates)]
holdout_rows = vol_df[vol_df["date"].isin(holdout_dates)]
print(f"{len(vol_df)} (date, asset) rows | train {len(train_rows)} | holdout {len(holdout_rows)}")

vol_model = GradientBoostingRegressor(max_depth=3, n_estimators=200, learning_rate=0.05, random_state=42)
vol_model.fit(train_rows[FEATURE_COLS], train_rows["label"])
pred = vol_model.predict(holdout_rows[FEATURE_COLS])

mae_ml = np.abs(pred - holdout_rows["label"]).mean()
mae_ewma = np.abs(holdout_rows["ewma_baseline"] - holdout_rows["label"]).mean()
print(f"\nForecast quality on held-out (date,asset) rows, still entirely pre-decision_start_date:")
print(f"  MAE, ML model:    {mae_ml:.4f}")
print(f"  MAE, EWMA(span={cfg.ewma_span}): {mae_ewma:.4f}")
print(f"  ML {'beats' if mae_ml < mae_ewma else 'does NOT beat'} the EWMA baseline on pure forecast error.")


100548 (date, asset) rows | train 70364 | holdout 30184



Forecast quality on held-out (date,asset) rows, still entirely pre-decision_start_date:
  MAE, ML model:    0.0783
  MAE, EWMA(span=60): 0.0665
  ML does NOT beat the EWMA baseline on pure forecast error.


In [7]:
fig = go.Figure()
sample_asset = prices.columns[0]
sample = holdout_rows[holdout_rows["asset"] == sample_asset].sort_values("date")
fig.add_trace(go.Scatter(x=sample["date"], y=sample["label"], name="Realized (forward 21d vol)", mode="lines"))
fig.add_trace(go.Scatter(x=sample["date"], y=vol_model.predict(sample[FEATURE_COLS]), name="ML prediction", mode="lines"))
fig.add_trace(go.Scatter(x=sample["date"], y=sample["ewma_baseline"], name="EWMA baseline", mode="lines"))
fig.update_layout(title=f"Vol forecast comparison — {sample_asset} (held-out period)", yaxis_title="Annualized vol", hovermode="x unified")
fig.show()


In [8]:
# Refit on the FULL pre-decision_start_date window (train+holdout) for the downstream backtest,
# now that architecture/hyperparams are validated above.
vol_model_final = GradientBoostingRegressor(max_depth=3, n_estimators=200, learning_rate=0.05, random_state=42)
vol_model_final.fit(vol_df[FEATURE_COLS], vol_df["label"])

def make_ml_vol_fn(model):
    def ml_vol_fn(train_data, span):
        feats = {f"rvol_{w}": train_data.tail(w).std() * np.sqrt(252) for w in FEATURE_WINDOWS}
        feats["abs_ret_1"] = train_data.iloc[-1].abs()
        feats["sq_ret_1"] = train_data.iloc[-1] ** 2
        X = pd.DataFrame(feats)[FEATURE_COLS]
        return pd.Series(model.predict(X), index=train_data.columns).clip(lower=1e-4)
    return ml_vol_fn

exp1_results, exp1_weights, exp1_metrics = run_and_score(
    eval_slice, cfg, "Experiment 1: ML vol forecast", vol_fn=make_ml_vol_fn(vol_model_final)
)
print(f"\nBaseline Sharpe {BASELINE_SHARPE:.3f}  ->  Experiment 1 Sharpe {exp1_metrics['strategy']['sharpe']:.3f} "
      f"({'improvement' if exp1_metrics['strategy']['sharpe'] > BASELINE_SHARPE else 'no improvement'})")


[Experiment 1: ML vol forecast] (14.3s) Sharpe 0.478 | MaxDD -12.44% | AnnRet 10.89% | AnnVol 10.21% | Turnover 0.31%

Baseline Sharpe 0.562  ->  Experiment 1 Sharpe 0.478 (no improvement)


---
# Experiment 2: HRP Bisection Weighting (Learned Top-Level Split)

**Idea:** `src/hrp.py`'s `_bisect()` splits each cluster's weight by a fixed
formula: `alloc_left = 1 - var_left/(var_left + var_right)` (inverse-variance
style). This experiment learns that ratio instead, **only at the top-level
split** of the dendrogram (deeper recursive splits keep using the original
production formula via `hrp_module._bisect`, minimizing how much of `_hrp.py`
gets reimplemented).

**Label construction (training only, strictly pre-`decision_start_date`):**
at each historical rebalance point, grid-search the split ratio that would
have maximized the blended cluster Sharpe over the *subsequent* 21 days —
this "looks forward" only within the training period, never past
`decision_start_date`, the same discipline as Experiment 1.

**Caveat:** grid-searching a hindsight-optimal label and then fitting a model
to predict it is a well-worn ML pattern, but with a small effective sample
size (~75 historical splits here) it's easy for the model to just memorize
noise. Treat this experiment's result skeptically even more than the others.


In [9]:
def hrp_split_features(sub_cov, left_idx, right_idx):
    var_left = hrp_module._cluster_var(sub_cov, left_idx)
    var_right = hrp_module._cluster_var(sub_cov, right_idx)
    ratio = var_left / (var_left + var_right) if (var_left + var_right) > 0 else 0.5
    return [var_left, var_right, ratio, len(left_idx), len(right_idx)]

ALLOC_GRID = np.linspace(0.1, 0.9, 9)

split_rows = []
j = WARMUP_ROWS
while j + 21 <= len(ml_train_returns):
    train_data = ml_train_returns.iloc[j - WARMUP_ROWS:j]
    forward = ml_train_returns.iloc[j:j + 21]

    vol = ewma_vol(train_data, span=cfg.ewma_span)
    corr = ewmc_corr(train_data, span=cfg.corr_span)
    corr = shrink_correlation(corr, cfg.corr_shrinkage)
    cov = build_covariance(vol, corr)
    cov = psd_repair(cov)

    d = hrp_module._distance_matrix(corr)
    order = leaves_list(linkage(squareform(d.values, checks=False), method="single"))
    items = [cov.index[i] for i in order]
    sub_cov = cov.values[np.ix_(order, order)]
    split = len(items) // 2
    left_idx, right_idx = list(range(split)), list(range(split, len(items)))
    left_items, right_items = items[:split], items[split:]

    best_alloc, best_score = 0.5, -np.inf
    for a in ALLOC_GRID:
        w_left = hrp_module._bisect(sub_cov[np.ix_(left_idx, left_idx)], left_items)
        w_right = hrp_module._bisect(sub_cov[np.ix_(right_idx, right_idx)], right_items)
        blended = {k: v * a for k, v in w_left.items()} | {k: v * (1 - a) for k, v in w_right.items()}
        blended_returns = forward[list(blended.keys())].dot(pd.Series(blended))
        s = blended_returns.mean() / blended_returns.std() if blended_returns.std() > 0 else -np.inf
        if s > best_score:
            best_score, best_alloc = s, a

    split_rows.append(hrp_split_features(sub_cov, left_idx, right_idx) + [best_alloc])
    j += 21

split_df = pd.DataFrame(split_rows, columns=["var_left", "var_right", "ratio", "size_left", "size_right", "best_alloc"])
print(f"{len(split_df)} historical top-level splits sampled for training")
split_df.describe()


77 historical top-level splits sampled for training


,var_left,var_right,ratio,size_left,size_right,best_alloc
count,77.0000,77.0000,77.0000,77.0000,77.0000,77.0000
mean,0.0164,0.0364,0.2831,24.0000,25.0000,0.5416
std,0.0298,0.0482,0.0844,0.0000,0.0000,0.3704
min,0.0025,0.0066,0.1462,24.0000,25.0000,0.1000
25%,0.0055,0.0154,0.2218,24.0000,25.0000,0.1000
50%,0.0082,0.0224,0.2632,24.0000,25.0000,0.7000
75%,0.0134,0.0391,0.3394,24.0000,25.0000,0.9000
max,0.2109,0.3486,0.5564,24.0000,25.0000,0.9000


In [10]:
SPLIT_FEATURE_COLS = ["var_left", "var_right", "ratio", "size_left", "size_right"]
split_model = GradientBoostingRegressor(max_depth=2, n_estimators=100, learning_rate=0.05, random_state=42)
split_model.fit(split_df[SPLIT_FEATURE_COLS], split_df["best_alloc"])

naive_mae = np.abs(split_df["ratio"] - split_df["best_alloc"]).mean()
print(f"MAE of the current fixed-formula ratio vs. the hindsight-optimal label (in-sample, for context only): {naive_mae:.3f}")

def make_custom_allocate(model):
    def custom_allocate(cov, corr, config):
        d = hrp_module._distance_matrix(corr)
        order = leaves_list(linkage(squareform(d.values, checks=False), method="single"))
        items = [cov.index[i] for i in order]
        sub_cov = cov.values[np.ix_(order, order)]
        split = len(items) // 2
        left_idx, right_idx = list(range(split)), list(range(split, len(items)))
        left_items, right_items = items[:split], items[split:]

        feats = hrp_split_features(sub_cov, left_idx, right_idx)
        alloc_left = float(np.clip(model.predict([feats])[0], 0.05, 0.95))

        w_left = hrp_module._bisect(sub_cov[np.ix_(left_idx, left_idx)], left_items)
        w_right = hrp_module._bisect(sub_cov[np.ix_(right_idx, right_idx)], right_items)
        weights = {k: v * alloc_left for k, v in w_left.items()} | {k: v * (1 - alloc_left) for k, v in w_right.items()}
        weights = hrp_module._clip_constraints(weights, config.min_asset_weight, config.max_asset_weight)
        total = sum(weights.values())
        return {k: v / total for k, v in weights.items()}
    return custom_allocate

exp2_results, exp2_weights, exp2_metrics = run_and_score(
    eval_slice, cfg, "Experiment 2: learned top-level HRP split", allocate_fn=make_custom_allocate(split_model)
)
print(f"\nBaseline Sharpe {BASELINE_SHARPE:.3f}  ->  Experiment 2 Sharpe {exp2_metrics['strategy']['sharpe']:.3f} "
      f"({'improvement' if exp2_metrics['strategy']['sharpe'] > BASELINE_SHARPE else 'no improvement'})")


MAE of the current fixed-formula ratio vs. the hindsight-optimal label (in-sample, for context only): 0.404


[Experiment 2: learned top-level HRP split] (13.9s) Sharpe 0.557 | MaxDD -11.53% | AnnRet 11.97% | AnnVol 10.71% | Turnover 0.35%

Baseline Sharpe 0.562  ->  Experiment 2 Sharpe 0.557 (no improvement)


---
# Experiment 3: Dynamic Universe Selection

**Idea:** instead of HRP always allocating across the full static 49-name
universe, narrow it each rebalance to the assets a classifier scores highest
on backward-looking momentum/vol features, then run HRP on that subset.

**Important caveat, more than the other three experiments:** this is
momentum-based stock *selection*, the closest thing here to a return
forecast — the category flagged earlier as highest-risk, both because it's
easiest to leak and because it risks just rediscovering the well-known
momentum factor rather than adding anything specific to this pipeline. Read
this result with real skepticism, more than Experiments 1 and 2.


In [11]:
mom_3m = prices.pct_change(63)
mom_6m = prices.pct_change(126)
mom_12m = prices.pct_change(252)
rvol_60 = log_returns_full.rolling(60).std() * np.sqrt(252)
fwd_ret_21 = log_returns_full.rolling(21).sum().shift(-21)

def stack(df, name):
    return df.rename_axis(index="date", columns="asset").stack().rename(name)

uni_df = pd.concat([
    stack(mom_3m, "mom_3m"), stack(mom_6m, "mom_6m"), stack(mom_12m, "mom_12m"),
    stack(rvol_60, "rvol_60"), stack(fwd_ret_21, "fwd_ret_21"),
], axis=1).dropna().reset_index()

uni_df = uni_df[uni_df["date"] < decision_start_date]  # training only, strictly pre-cutoff
uni_df["rank"] = uni_df.groupby("date")["fwd_ret_21"].rank(pct=True)
uni_df["label"] = (uni_df["rank"] >= 0.5).astype(int)

# non-overlapping 21-day sampling to avoid inflating the effective sample size with near-duplicate rows
sampled_dates = sorted(uni_df["date"].unique())[::21]
uni_train = uni_df[uni_df["date"].isin(sampled_dates)]
print(f"{len(uni_train)} training rows across {len(sampled_dates)} sampled dates, "
      f"label balance: {uni_train['label'].mean():.2%} positive")

UNI_FEATURE_COLS = ["mom_3m", "mom_6m", "mom_12m", "rvol_60"]
universe_model = GradientBoostingClassifier(max_depth=3, n_estimators=150, learning_rate=0.05, random_state=42)
universe_model.fit(uni_train[UNI_FEATURE_COLS], uni_train["label"])


4410 training rows across 90 sampled dates, label balance: 51.02% positive


,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",150
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (i

In [12]:
UNIVERSE_FRACTION = 0.6  # keep the top 60% of assets by predicted score each rebalance

def ml_universe_fn(train_data, config, as_of_date):
    last_date = train_data.index[-1]
    feats = pd.DataFrame({
        "mom_3m": prices.loc[last_date] / prices.loc[:last_date].iloc[-64] - 1 if len(prices.loc[:last_date]) > 63 else np.nan,
        "mom_6m": prices.loc[last_date] / prices.loc[:last_date].iloc[-127] - 1 if len(prices.loc[:last_date]) > 126 else np.nan,
        "mom_12m": prices.loc[last_date] / prices.loc[:last_date].iloc[-253] - 1 if len(prices.loc[:last_date]) > 252 else np.nan,
        "rvol_60": train_data.tail(60).std() * np.sqrt(252),
    }).dropna()
    if len(feats) < 5:
        return list(train_data.columns)  # not enough history to score — fall back to full universe
    scores = pd.Series(universe_model.predict_proba(feats[UNI_FEATURE_COLS])[:, 1], index=feats.index)
    n_keep = max(5, int(len(scores) * UNIVERSE_FRACTION))
    return list(scores.sort_values(ascending=False).head(n_keep).index)

exp3_results, exp3_weights, exp3_metrics = run_and_score(
    eval_slice, cfg, "Experiment 3: ML-narrowed dynamic universe", universe_fn=ml_universe_fn
)
print(f"\nBaseline Sharpe {BASELINE_SHARPE:.3f}  ->  Experiment 3 Sharpe {exp3_metrics['strategy']['sharpe']:.3f} "
      f"({'improvement' if exp3_metrics['strategy']['sharpe'] > BASELINE_SHARPE else 'no improvement'})")


[Experiment 3: ML-narrowed dynamic universe] (5.3s) Sharpe 0.318 | MaxDD -11.50% | AnnRet 9.37% | AnnVol 10.60% | Turnover 20.99%

Baseline Sharpe 0.562  ->  Experiment 3 Sharpe 0.318 (no improvement)


---
# Experiment 4: Hyperparameter Search Over Existing Knobs

**Idea:** not a new model — a walk-forward-validated random search over
`vol_target_annual`, `rebalance_band`, `weight_smoothing`, `corr_shrinkage`,
`ewma_span` (all currently hand-picked constants in `config.py`), scored on a
**search window**, with the winning combination then validated on a
**separate, later, non-overlapping holdout window** — proper hygiene against
overfitting the search to one period. This is the lowest-risk lever of the
four since it changes no modeling logic, only which point in an already-existing
parameter space is used.


In [13]:
SEARCH_TEST_YEARS, HOLDOUT_TEST_YEARS = 2, 1
holdout_rows_n = HOLDOUT_TEST_YEARS * 252
search_rows_n = SEARCH_TEST_YEARS * 252

holdout_slice = prices.tail(WARMUP_ROWS + holdout_rows_n)
search_start = len(prices) - holdout_rows_n - search_rows_n - WARMUP_ROWS
search_slice = prices.iloc[search_start: len(prices) - holdout_rows_n]

print(f"search_slice:  {search_slice.index.min().date()} to {search_slice.index.max().date()}")
print(f"holdout_slice: {holdout_slice.index.min().date()} to {holdout_slice.index.max().date()}  (strictly after search_slice)")

def quick_sharpe(prices_slice, config):
    strat = Strategy(train_window=config.train_years * 252, test_window=config.test_months * 21,
                      transaction_cost=config.transaction_cost_bps / 10000)
    strat.config = config
    results, _, _ = strat.run(prices_slice)
    return sharpe_ratio(results["Strategy"].dropna(), config.risk_free_rate_annual)

rng = np.random.default_rng(42)
N_TRIALS = 15
trials = []
t0 = time.time()
for trial in range(N_TRIALS):
    candidate = replace(
        cfg,
        vol_target_annual=float(rng.uniform(0.06, 0.16)),
        rebalance_band=float(rng.uniform(0.02, 0.20)),
        weight_smoothing=float(rng.uniform(0.10, 0.60)),
        corr_shrinkage=float(rng.uniform(0.0, 0.30)),
        ewma_span=int(rng.choice([20, 40, 60, 90, 120])),
    )
    s = quick_sharpe(search_slice, candidate)
    trials.append({"trial": trial, "sharpe": s, "vol_target_annual": candidate.vol_target_annual,
                    "rebalance_band": candidate.rebalance_band, "weight_smoothing": candidate.weight_smoothing,
                    "corr_shrinkage": candidate.corr_shrinkage, "ewma_span": candidate.ewma_span})
    print(f"  trial {trial+1}/{N_TRIALS}: Sharpe {s:.3f}  ({time.time()-t0:.0f}s elapsed)")

trials_df = pd.DataFrame(trials).sort_values("sharpe", ascending=False)
trials_df


search_slice:  2021-08-10 to 2025-09-05
holdout_slice: 2023-08-22 to 2026-09-11  (strictly after search_slice)


  trial 1/15: Sharpe 0.918  (9s elapsed)


  trial 2/15: Sharpe 0.863  (19s elapsed)


  trial 3/15: Sharpe 0.902  (28s elapsed)


  trial 4/15: Sharpe 0.969  (38s elapsed)


...

  trial 12/15: Sharpe 0.946  (118s elapsed)


  trial 13/15: Sharpe 0.902  (128s elapsed)


  trial 14/15: Sharpe 0.811  (137s elapsed)


  trial 15/15: Sharpe 1.003  (146s elapsed)


,trial,sharpe,vol_target_annual,rebalance_band,weight_smoothing,corr_shrinkage,ewma_span
14,14,1.0029,0.1365,0.1342,0.3768,0.1678,90
8,8,1.0011,0.0730,0.1056,0.2135,0.2009,120
7,7,0.9887,0.0926,0.0867,0.3348,0.0568,120
3,3,0.9694,0.1043,0.0609,0.3773,0.0191,120
10,10,0.9688,0.1405,0.0897,0.2442,0.2047,90
11,11,0.9465,0.0800,0.0213,0.4935,0.1995,20
5,5,0.9454,0.1493,0.1601,0.1973,0.1400,120
6,6,0.9330,0.0644,0.0478,0.4415,0.2234,40
9,9,0.9305,0.1433,0.1460,0.2562,0.2497,60
0,0,0.9182,0.1374,0.0990,0.5293,0.2092,40


In [14]:
best = trials_df.iloc[0]
default_search_sharpe = quick_sharpe(search_slice, cfg)
print(f"Default config on search_slice:  Sharpe {default_search_sharpe:.3f}")
print(f"Best of {N_TRIALS} random trials on search_slice: Sharpe {best['sharpe']:.3f}")
print(f"  vol_target_annual={best.vol_target_annual:.3f}, rebalance_band={best.rebalance_band:.3f}, "
      f"weight_smoothing={best.weight_smoothing:.3f}, corr_shrinkage={best.corr_shrinkage:.3f}, ewma_span={int(best.ewma_span)}")

best_config = replace(cfg, vol_target_annual=best.vol_target_annual, rebalance_band=best.rebalance_band,
                       weight_smoothing=best.weight_smoothing, corr_shrinkage=best.corr_shrinkage, ewma_span=int(best.ewma_span))

# The real test: does the winning config still win on a LATER, never-searched window?
holdout_default_sharpe = quick_sharpe(holdout_slice, cfg)
holdout_best_sharpe = quick_sharpe(holdout_slice, best_config)

print(f"\n--- Holdout validation (period the search never saw) ---")
print(f"Default config on holdout:      Sharpe {holdout_default_sharpe:.3f}")
print(f"'Best' config on holdout:       Sharpe {holdout_best_sharpe:.3f}")
gap = best["sharpe"] - holdout_best_sharpe
print(f"\nSearch-to-holdout gap: {gap:.3f} Sharpe "
      f"({'looks like overfitting to the search window — treat the \'best\' config with suspicion' if gap > 0.3 else 'reasonably consistent, more encouraging'})")


Default config on search_slice:  Sharpe 0.953
Best of 15 random trials on search_slice: Sharpe 1.003
  vol_target_annual=0.136, rebalance_band=0.134, weight_smoothing=0.377, corr_shrinkage=0.168, ewma_span=90



--- Holdout validation (period the search never saw) ---
Default config on holdout:      Sharpe -0.233
'Best' config on holdout:       Sharpe -0.292

Search-to-holdout gap: 1.295 Sharpe (looks like overfitting to the search window — treat the 'best' config with suspicion)


---
# Experiment 5: Regime-Adaptive Overlay

**Idea:** rather than a fixed `vol_target_annual`/`rebalance_band` for the
whole backtest, classify each rebalance's market regime (calm vs. turbulent)
from backward-looking features, and adjust just those two overlay knobs
accordingly. This only touches the vol-target/banding overlay — the
covariance estimate and the HRP allocator itself are untouched, the same
"bounded blast radius" reasoning that made Experiment 2 the closest thing to
neutral so far.

**Design choice, deliberately:** the regime *classifier* (KMeans on trailing
portfolio vol + average pairwise correlation) is fit on pre-cutoff data, but
the regime→knob **mapping uses fixed multipliers, not another hindsight grid
search**. Experiment 4 just showed exactly how easily grid-searching a small
number of historical regime windows overfits — so this experiment isolates
one question: does *knowing the regime* help at all, using an economically
motivated adjustment (de-risk and widen bands when turbulent; lean in and
tighten bands when calm), rather than "did we get lucky picking multiplier
values." A follow-up could search the multipliers properly with a much larger
sample of regime occurrences, but that's out of scope here.


In [15]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

REGIME_FEATURES = ["port_vol", "avg_corr"]
regime_rows = []
j = WARMUP_ROWS
while j <= len(ml_train_returns):
    train_data = ml_train_returns.iloc[j - WARMUP_ROWS:j]
    vol = ewma_vol(train_data, span=cfg.ewma_span)
    corr = ewmc_corr(train_data, span=cfg.corr_span)
    port_vol = float(vol.mean())
    avg_corr = float(corr.values[np.triu_indices_from(corr.values, k=1)].mean())
    regime_rows.append({"date": train_data.index[-1], "port_vol": port_vol, "avg_corr": avg_corr})
    j += 21

regime_df = pd.DataFrame(regime_rows)
scaler = StandardScaler().fit(regime_df[REGIME_FEATURES])
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10).fit(scaler.transform(regime_df[REGIME_FEATURES]))
regime_df["cluster"] = kmeans.labels_

cluster_summary = regime_df.groupby("cluster")[REGIME_FEATURES].mean()
high_vol_cluster = int(cluster_summary["port_vol"].idxmax())
print(cluster_summary)
print(f"\nHigh-vol cluster = {high_vol_cluster} | regime counts: {regime_df['cluster'].value_counts().to_dict()}")


         port_vol  avg_corr
cluster                    
0          0.2866    0.2166
1          0.6737    0.5059

High-vol cluster = 1 | regime counts: {0: 74, 1: 4}


In [16]:
REGIME_MULTIPLIERS = {
    "high_vol": {"vol_target_mult": 0.7, "band_mult": 1.3},   # de-risk, trade less often amid noise
    "low_vol":  {"vol_target_mult": 1.3, "band_mult": 0.7},   # lean in, rebalance more precisely
}

def make_regime_overlay_fn(scaler, kmeans, high_vol_cluster, base_config):
    def overlay_fn(train_data, config):
        vol = ewma_vol(train_data, span=config.ewma_span)
        corr = ewmc_corr(train_data, span=config.corr_span)
        port_vol = float(vol.mean())
        avg_corr = float(corr.values[np.triu_indices_from(corr.values, k=1)].mean())
        cluster = int(kmeans.predict(scaler.transform([[port_vol, avg_corr]]))[0])
        regime = "high_vol" if cluster == high_vol_cluster else "low_vol"
        mult = REGIME_MULTIPLIERS[regime]
        return base_config.vol_target_annual * mult["vol_target_mult"], base_config.rebalance_band * mult["band_mult"]
    return overlay_fn

exp5_results, exp5_weights, exp5_metrics = run_and_score(
    eval_slice, cfg, "Experiment 5: regime-adaptive overlay (KMeans)",
    overlay_fn=make_regime_overlay_fn(scaler, kmeans, high_vol_cluster, cfg),
)
print(f"\nBaseline Sharpe {BASELINE_SHARPE:.3f}  ->  Experiment 5 Sharpe {exp5_metrics['strategy']['sharpe']:.3f} "
      f"({'improvement' if exp5_metrics['strategy']['sharpe'] > BASELINE_SHARPE else 'no improvement'})")


[Experiment 5: regime-adaptive overlay (KMeans)] (29.9s) Sharpe 0.550 | MaxDD -15.46% | AnnRet 13.57% | AnnVol 13.75% | Turnover 0.35%

Baseline Sharpe 0.562  ->  Experiment 5 Sharpe 0.550 (no improvement)


## Experiment 5b: Same Idea, Threshold Rule Instead of KMeans

KMeans found a **74 vs. 4** split on `regime_df` above — one anomalous
stretch, not a genuine calm/turbulent split, which meant the "lean in" 1.3×
multiplier fired on almost every rebalance and mostly just ran the strategy
hotter rather than adapting to anything. The likely fix isn't a fancier
classifier — it's that 78 sampled points is too few for unsupervised
clustering to find a balanced split reliably. A **median-threshold rule**
(computed once from the same pre-cutoff `regime_df`, so still trained only on
pre-`decision_start_date` data) guarantees an even split by construction and
tests the same underlying idea — does *knowing you're in an elevated-vol
period* help — without the clustering's failure mode.


In [17]:
vol_threshold = float(regime_df["port_vol"].median())
print(f"Median portfolio vol (pre-cutoff training period): {vol_threshold:.2%}")
print(f"Split under this threshold: {(regime_df['port_vol'] > vol_threshold).sum()} high-vol vs "
      f"{(regime_df['port_vol'] <= vol_threshold).sum()} low-vol windows")

def make_threshold_overlay_fn(threshold, base_config):
    def overlay_fn(train_data, config):
        vol = ewma_vol(train_data, span=config.ewma_span)
        port_vol = float(vol.mean())
        regime = "high_vol" if port_vol > threshold else "low_vol"
        mult = REGIME_MULTIPLIERS[regime]
        return base_config.vol_target_annual * mult["vol_target_mult"], base_config.rebalance_band * mult["band_mult"]
    return overlay_fn

exp5b_results, exp5b_weights, exp5b_metrics = run_and_score(
    eval_slice, cfg, "Experiment 5b: regime-adaptive overlay (median threshold)",
    overlay_fn=make_threshold_overlay_fn(vol_threshold, cfg),
)
print(f"\nBaseline Sharpe {BASELINE_SHARPE:.3f}  ->  Experiment 5b Sharpe {exp5b_metrics['strategy']['sharpe']:.3f} "
      f"({'improvement' if exp5b_metrics['strategy']['sharpe'] > BASELINE_SHARPE else 'no improvement'})")


Median portfolio vol (pre-cutoff training period): 27.64%
Split under this threshold: 39 high-vol vs 39 low-vol windows


[Experiment 5b: regime-adaptive overlay (median threshold)] (14.4s) Sharpe 0.510 | MaxDD -15.46% | AnnRet 12.88% | AnnVol 13.49% | Turnover 0.35%

Baseline Sharpe 0.562  ->  Experiment 5b Sharpe 0.510 (no improvement)


---
# Experiment 6: Volatility Shrinkage Blend

**Idea:** the same trick `shrink_correlation()` already applies to the
correlation matrix — blend the noisy fast estimate toward a calmer, more
stable target — applied to volatility instead. Right now `ewma_vol` (a
60-day exponential average) is trusted 100%; this blends it with a slower,
250-day EWMA that doesn't overreact to a recent spike:

```
blended_vol = (1 - delta) * ewma_vol(span=60) + delta * ewma_vol(span=250)
```

**Not really "ML"** — no training set, no fitting step, nothing that can
overfit the way a gradient-boosted model can. One smooth parameter (`delta`)
picked by a small grid search, which is a fundamentally lower-risk search
than Experiment 4's 5-parameter, 15-trial random search over loosely coupled
knobs — far less room for a smooth 1D curve to spike on noise.

**Caveat on windows:** `delta` is picked by grid search on `search_slice`
(defined in Experiment 4) — whose test period is a *subset* of `eval_slice`'s,
so this isn't a fully disjoint holdout the way Experiment 4's was. Given how
constrained this search is (1 parameter, 6 grid points, smooth function)
that's a much smaller concern than it would be for a high-flexibility model,
but worth naming rather than glossing over.


In [18]:
LONG_RUN_SPAN = 250  # ~1 trading year — deliberately slow, not fit to anything

def make_vol_blend_fn(delta, long_run_span=LONG_RUN_SPAN):
    def vol_blend_fn(train_data, span):
        fast = ewma_vol(train_data, span=span)
        slow = ewma_vol(train_data, span=long_run_span)
        return (1 - delta) * fast + delta * slow
    return vol_blend_fn

delta_grid = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]  # extended past 0.5 since the first pass plateaued at the edge of the grid rather than turning back down
delta_trials = []
for delta in delta_grid:
    results, _, _ = walk_forward(search_slice, cfg, vol_fn=make_vol_blend_fn(delta))
    s = sharpe_ratio(results["Strategy"].dropna(), cfg.risk_free_rate_annual)
    delta_trials.append({"delta": delta, "sharpe": s})
    print(f"  delta={delta:.1f}: Sharpe {s:.3f}  (on search_slice)")

delta_df = pd.DataFrame(delta_trials)
best_delta = float(delta_df.sort_values("sharpe", ascending=False).iloc[0]["delta"])
print(f"\nBest delta on search_slice: {best_delta:.1f}")
print("Curve shape (should be smooth, not spiky, if this is real signal rather than noise):")
delta_df


  delta=0.0: Sharpe 0.953  (on search_slice)


  delta=0.1: Sharpe 0.954  (on search_slice)


  delta=0.2: Sharpe 0.956  (on search_slice)


  delta=0.3: Sharpe 0.974  (on search_slice)


  delta=0.4: Sharpe 0.974  (on search_slice)


...

  delta=0.7: Sharpe 0.921  (on search_slice)


  delta=0.8: Sharpe 0.924  (on search_slice)


  delta=0.9: Sharpe 0.945  (on search_slice)


  delta=1.0: Sharpe 0.941  (on search_slice)

Best delta on search_slice: 0.5
Curve shape (should be smooth, not spiky, if this is real signal rather than noise):


,delta,sharpe
0,0.0000,0.9528
1,0.1000,0.9542
2,0.2000,0.9556
3,0.3000,0.9745
4,0.4000,0.9745
5,0.5000,0.9745
6,0.6000,0.9652
7,0.7000,0.9211
8,0.8000,0.9237
9,0.9000,0.9452


In [19]:
fig = go.Figure(go.Scatter(x=delta_df["delta"], y=delta_df["sharpe"], mode="lines+markers"))
fig.update_layout(title="Sharpe vs. shrinkage delta (search_slice)", xaxis_title="delta (0 = pure EWMA, 1 = pure long-run avg)", yaxis_title="Sharpe")
fig.show()

exp6_results, exp6_weights, exp6_metrics = run_and_score(
    eval_slice, cfg, f"Experiment 6: vol shrinkage blend (delta={best_delta:.1f})",
    vol_fn=make_vol_blend_fn(best_delta),
)
print(f"\nBaseline Sharpe {BASELINE_SHARPE:.3f}  ->  Experiment 6 Sharpe {exp6_metrics['strategy']['sharpe']:.3f} "
      f"({'improvement' if exp6_metrics['strategy']['sharpe'] > BASELINE_SHARPE else 'no improvement'})")


[Experiment 6: vol shrinkage blend (delta=0.5)] (14.7s) Sharpe 0.583 | MaxDD -11.28% | AnnRet 12.17% | AnnVol 10.58% | Turnover 0.23%

Baseline Sharpe 0.562  ->  Experiment 6 Sharpe 0.583 (improvement)


## Experiment 6 holdout check

`search_slice`'s test window is a subset of `eval_slice`'s, so the result
above isn't a fully disjoint out-of-sample check. `holdout_slice` (from
Experiment 4) never entered the delta search at all — a genuinely
never-seen-by-this-experiment window, and the strategy's roughest recent
stretch (see Experiment 4's holdout numbers). If the shrinkage blend still
helps *here*, that's a real signal, not a search-window artifact.


In [20]:
holdout_default_vol_sharpe = quick_sharpe(holdout_slice, cfg)
holdout_default_results, _, _ = walk_forward(holdout_slice, cfg, vol_fn=make_vol_blend_fn(best_delta))
holdout_blend_sharpe = sharpe_ratio(holdout_default_results["Strategy"].dropna(), cfg.risk_free_rate_annual)

print(f"holdout_slice ({holdout_slice.index.min().date()} to {holdout_slice.index.max().date()}):")
print(f"  Default (EWMA-only) vol:        Sharpe {holdout_default_vol_sharpe:.3f}")
print(f"  Vol shrinkage blend (delta={best_delta:.1f}): Sharpe {holdout_blend_sharpe:.3f}")
print(f"  Difference: {holdout_blend_sharpe - holdout_default_vol_sharpe:+.3f}")


holdout_slice (2023-08-22 to 2026-09-11):
  Default (EWMA-only) vol:        Sharpe -0.233
  Vol shrinkage blend (delta=0.5): Sharpe -0.252
  Difference: -0.019


---
# Summary

Same baseline, same `eval_slice`/comparable windows, one change at a time.


In [21]:
summary = pd.DataFrame([
    {"Experiment": "Baseline", "Sharpe": BASELINE_SHARPE, "MaxDD": baseline_metrics["strategy"]["max_drawdown"], "AnnRet": baseline_metrics["strategy"]["annualized_return"]},
    {"Experiment": "1. ML vol forecast", "Sharpe": exp1_metrics["strategy"]["sharpe"], "MaxDD": exp1_metrics["strategy"]["max_drawdown"], "AnnRet": exp1_metrics["strategy"]["annualized_return"]},
    {"Experiment": "2. Learned HRP split", "Sharpe": exp2_metrics["strategy"]["sharpe"], "MaxDD": exp2_metrics["strategy"]["max_drawdown"], "AnnRet": exp2_metrics["strategy"]["annualized_return"]},
    {"Experiment": "3. Dynamic universe", "Sharpe": exp3_metrics["strategy"]["sharpe"], "MaxDD": exp3_metrics["strategy"]["max_drawdown"], "AnnRet": exp3_metrics["strategy"]["annualized_return"]},
    {"Experiment": "4. Best hyperparams (holdout)", "Sharpe": holdout_best_sharpe, "MaxDD": None, "AnnRet": None},
    {"Experiment": "5. Regime-adaptive overlay (KMeans)", "Sharpe": exp5_metrics["strategy"]["sharpe"], "MaxDD": exp5_metrics["strategy"]["max_drawdown"], "AnnRet": exp5_metrics["strategy"]["annualized_return"]},
    {"Experiment": "5b. Regime-adaptive overlay (threshold)", "Sharpe": exp5b_metrics["strategy"]["sharpe"], "MaxDD": exp5b_metrics["strategy"]["max_drawdown"], "AnnRet": exp5b_metrics["strategy"]["annualized_return"]},
    {"Experiment": "6. Vol shrinkage blend", "Sharpe": exp6_metrics["strategy"]["sharpe"], "MaxDD": exp6_metrics["strategy"]["max_drawdown"], "AnnRet": exp6_metrics["strategy"]["annualized_return"]},
])
summary["Sharpe_vs_baseline"] = summary["Sharpe"] - BASELINE_SHARPE
summary


,Experiment,Sharpe,MaxDD,AnnRet,Sharpe_vs_baseline
0,Baseline,0.5616,-0.1156,0.1202,0.0000
1,1. ML vol forecast,0.4784,-0.1244,0.1089,-0.0832
2,2. Learned HRP split,0.5572,-0.1153,0.1197,-0.0044
3,3. Dynamic universe,0.3179,-0.1150,0.0937,-0.2437
4,4. Best hyperparams (holdout),-0.2917,NaN,NaN,-0.8533
5,5. Regime-adaptive overlay (KMeans),0.5502,-0.1546,0.1357,-0.0114
6,5b. Regime-adaptive overlay (threshold),0.5097,-0.1546,0.1288,-0.0519
7,6. Vol shrinkage blend,0.5829,-0.1128,0.1217,0.0213


**Reading this table:** a positive `Sharpe_vs_baseline` on a *single* short
window is a reason to investigate further, not a reason to ship — none of
these ran walk-forward-retrained, none were tested across multiple
non-overlapping windows, and Experiment 3 in particular carries real leakage
risk if extended carelessly. Whichever (if any) look promising, the next step
before touching `src/` would be re-running that one experiment across several
different `EXPERIMENT_TEST_YEARS` windows to see if the improvement holds up
or was specific to this particular stretch of history.
